# Audio8 ASR Infinite: speech to text

Transcribes an audio file with [Edge0/Audio8-ASR-Infinite](https://huggingface.co/Edge0/Audio8-ASR-Infinite) (Chinese and English), loaded straight from the Hugging Face Hub.

- **Colab:** Runtime > Change runtime type > GPU, then Runtime > Run all.
- The first run downloads about 8 GB of model weights.
- Audio is decoded as *simulated streaming* from a file (no microphone input). Keep clips to about 30 seconds, the model's native context.

In [ ]:
%pip install -q "audio8-asr-infinite @ git+https://github.com/Edge0-AI/Audio8-ASR-Infinite.git" librosa

In [ ]:
import subprocess

import librosa
import torch
from huggingface_hub import hf_hub_download
from transformers import AutoFeatureExtractor, AutoTokenizer

from audio8_asr_infinite.modeling.modeling_audio8_asr_infinite import (
    Audio8ASRInfiniteForConditionalGeneration,
    resolve_qwen_language_token_id,
    resolve_qwen_streaming_special_token_ids,
)
from audio8_asr_infinite.streaming_inference import simulated_streaming_greedy_decode_batch

CHECKPOINT = "Edge0/Audio8-ASR-Infinite"
LANGUAGE = "en"  # "en" or "zh"
TARGET_DELAY_MS = 480  # transcription delay: lower is faster, higher is more accurate
CLOCK_MS = 80  # audio clock, matches the 80 ms operating point
MAX_SECONDS = 30  # the model's native context

device = "cuda" if torch.cuda.is_available() else "cpu"
no_native_bf16 = device == "cuda" and not torch.cuda.is_bf16_supported(including_emulation=False)
dtype = torch.float16 if no_native_bf16 else torch.bfloat16
print(f"device={device}, dtype={dtype}")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT, trust_remote_code=True)
feature_extractor = AutoFeatureExtractor.from_pretrained(CHECKPOINT, trust_remote_code=True)
model = Audio8ASRInfiniteForConditionalGeneration.from_pretrained(
    CHECKPOINT, trust_remote_code=True, dtype=dtype
).eval().to(device)

In [ ]:
# Demo audio: the first 30 s of the model's own demo video, as 16 kHz mono wav.
# To transcribe your own file instead, set AUDIO_PATH to it (wav/mp3/flac) and skip the next 3 lines.
# In Colab you can upload one with: from google.colab import files; AUDIO_PATH = next(iter(files.upload()))
demo_video = hf_hub_download(CHECKPOINT, "Audio8-Asr-Infinite-Demo.mp4")
AUDIO_PATH = "sample.wav"
subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", demo_video, "-t", str(MAX_SECONDS), "-ac", "1", "-ar", "16000", AUDIO_PATH], check=True)

waveform, _ = librosa.load(AUDIO_PATH, sr=16000, mono=True, duration=MAX_SECONDS)
print(f"{len(waveform) / 16000:.1f} s of audio loaded")

In [ ]:
class AudioConfig:  # 80 ms audio clock at 16 kHz
    raw_audio_samples_per_token = 1280
    streaming_n_left_pad_tokens = 18
    sampling_rate = 16000


results = simulated_streaming_greedy_decode_batch(
    model=model,
    tokenizer=tokenizer,
    feature_extractor=feature_extractor,
    waveforms=[waveform],
    language_token_ids=[resolve_qwen_language_token_id(tokenizer, LANGUAGE)],
    special_ids=resolve_qwen_streaming_special_token_ids(tokenizer),
    audio_config=AudioConfig(),
    num_delay_tokens=[TARGET_DELAY_MS // CLOCK_MS],
    right_pad_text_tokens=10,
    dtype=dtype,
    device=next(model.parameters()).device,
)
print("Transcription:", results[0]["final_text"])